In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# import seaborn as sns
import cv2
# from sentry_sdk.utils import epoch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy import ndimage
import tensorflow as tf
import tkinter as tk
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


print("Toate pachetele au fost instalate cu succes!")

Toate pachetele au fost instalate cu succes!


In [ ]:
# mnist = tf.keras.datasets.mnist
# (x_train, y_train), (x_test, y_test) = mnist.load_data()
# 
# # Normalize the images to a range of 0 to 1
# x_train = tf.keras.utils.normalize(x_train, axis=1)
# x_test = tf.keras.utils.normalize(x_test, axis=1)
# 
# # Reshape the data to include a channel dimension
# model = tf.keras.models.Sequential()
# model.add(tf.keras.layers.Flatten(input_shape=(28, 28)))
# 
# # Add convolutional layers
# model.add(tf.keras.layers.Dense(128, activation='relu'))
# model.add(tf.keras.layers.Dense(128, activation='relu'))
# model.add(tf.keras.layers.Dense(10, activation='softmax'))
# 
# # Compile the model
# model.compile(optimizer='adam',
#               loss='sparse_categorical_crossentropy',
#               metrics=['accuracy'])
# 
# model.fit(x_train, y_train, epochs=3)
# 
# model.save('mnist.h5')

In [ ]:
# model = tf.keras.models.load_model('mnist.h5')
# loss, accuracy = model.evaluate(x_test, y_test)
# 
# print(loss, accuracy)

model ok, predictie proasta

In [ ]:
# Încarc setul de date MNIST
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalizez imaginile la intervalul [0, 1]
x_train = tf.keras.utils.normalize(x_train, axis=1)
x_test = tf.keras.utils.normalize(x_test, axis=1)

# Definirea funcției pentru calculul trăsăturilor geometrice relevante
def calculate_geometric_features(image):
    moments = cv2.moments(image)

    # Aria imaginii (numărul de pixeli albi)
    area = moments['m00']

    # Calculăm centrul de masă (cx, cy) al imaginii
    cx = moments['m10'] / area if area != 0 else 0
    cy = moments['m01'] / area if area != 0 else 0

    # Calculăm momentul de inerție pentru axa x (Ixx), axa y (Iyy) și momentul mixt (Ixy)
    Ixx = moments['mu20'] / area if area != 0 else 0
    Iyy = moments['mu02'] / area if area != 0 else 0
    Ixy = moments['mu11'] / area if area != 0 else 0

    # Calculăm unghiul de rotație al cifrei pe baza momentelor
    angle = 0.5 * np.arctan2(2 * Ixy, Ixx - Iyy)

    # Calculăm perimetrul conturului cifrei
    contours, _ = cv2.findContours(image.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    perimeter = cv2.arcLength(contours[0], True) if contours else 0

    # Calculăm subțierea (thinness ratio), care este un indicator al formei
    thinness = (4 * np.pi * area) / (perimeter ** 2) if perimeter != 0 else 0

    # Calculăm elongarea (aspect ratio) folosind dimensiunile imaginii
    x, y, w, h = cv2.boundingRect(image.astype(np.uint8))
    elongation = h / w if w != 0 else 0

    # Returnăm trăsăturile geometrice relevante
    return [area, cx, cy, angle, perimeter, thinness, elongation]

# Extragem trăsăturile geometrice pentru setul de antrenare și testare
geo_train = np.array([calculate_geometric_features(img) for img in x_train])
geo_test = np.array([calculate_geometric_features(img) for img in x_test])

# Scalez trăsăturile geometrice
scaler = StandardScaler()
geo_train = scaler.fit_transform(geo_train)
geo_test = scaler.transform(geo_test)

# Combinăm datele de imagine cu trăsăturile geometrice
x_train_combined = np.hstack((x_train.reshape(len(x_train), -1), geo_train))
x_test_combined = np.hstack((x_test.reshape(len(x_test), -1), geo_test))

# Construim modelul neuronal
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(x_train_combined.shape[1],)),  # Definim inputul explicit
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.3),  # Dropout pentru prevenirea overfitting-ului
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Compilăm modelul
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Antrenăm modelul
model.fit(x_train_combined, y_train, epochs=10)

# Evaluăm modelul
test_loss, test_acc = model.evaluate(x_test_combined, y_test)
print(f'Precizia pe setul de testare: {test_acc}')

# Salvăm modelul
model.save('mnist_geometric2.keras')



Epoch 1/10


In [ ]:
# from tkinter import Label
# from PIL import Image, ImageTk
# 
# class DrawingApp:
#     def __init__(self, root):
#         self.root = root
#         self.root.title("Desenează o cifră")
# 
#         # Canvas pentru desenat (dimensiune 280x280)
#         self.canvas = tk.Canvas(self.root, width=280, height=280, bg="white")
#         self.canvas.pack()
#         self.canvas.bind("<B1-Motion>", self.paint)
# 
#         # Imaginea stocată de 280x280
#         self.image = np.zeros((280, 280), dtype=np.uint8)
# 
#         # Butoane pentru acțiuni
#         self.predict_button = tk.Button(self.root, text="Recunoaște", command=self.recognize_digit)
#         self.predict_button.pack()
#         self.clear_button = tk.Button(self.root, text="Șterge", command=self.clear)
#         self.clear_button.pack()
# 
#         # Etichete pentru afișarea rezultatului
#         self.prediction_label = Label(self.root, text="Cifra prezisă: -", font=('Helvetica', 14))
#         self.prediction_label.pack()
# 
#         self.accuracy_label = Label(self.root, text="Acuratețe: -", font=('Helvetica', 14))
#         self.accuracy_label.pack()
# 
#         # Label pentru imagine
#         self.image_label = Label(self.root)
#         self.image_label.pack()
# 
#         # Încarc modelul salvat
#         self.model = tf.keras.models.load_model('mnist_geometric.keras')
# 
#     def paint(self, event):
#         x, y = event.x, event.y
#         if 0 <= x < 280 and 0 <= y < 280:
#             self.canvas.create_oval(x-5, y-5, x+5, y+5, fill="black", width=10)
#             cv2.circle(self.image, (x, y), 5, 255, -1)  # Cercul în imagine
# 
#     def clear(self):
#         self.canvas.delete("all")
#         self.image = np.zeros((280, 280), dtype=np.uint8)
# 
#     def calculate_geometric_features(self, image):
#         image_uint8 = image.astype(np.uint8)
#         moments = cv2.moments(image_uint8)
#         
#         area = moments['m00']
#         cx = moments['m10'] / area if area != 0 else 0
#         cy = moments['m01'] / area if area != 0 else 0
#         Ixx = moments['mu20'] / area if area != 0 else 0
#         Iyy = moments['mu02'] / area if area != 0 else 0
#         Ixy = moments['mu11'] / area if area != 0 else 0
#         angle = 0.5 * np.arctan2(2 * Ixy, Ixx - Iyy)
# 
#         contours, _ = cv2.findContours(image_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#         perimeter = cv2.arcLength(contours[0], True) if contours else 0
#         thinness = (4 * np.pi * area) / (perimeter ** 2) if perimeter != 0 else 0
#         x, y, w, h = cv2.boundingRect(image_uint8)
#         elongation = h / w if w != 0 else 0
# 
#         return [area, cx, cy, angle, perimeter, thinness, elongation]
# 
#     def recognize_digit(self):
#         # Redimensionăm la 28x28
#         img_resized = cv2.resize(self.image, (28, 28))
# 
#         # Aplatizăm imaginea (fără normalizare)
#         img_flat = img_resized.flatten().astype(np.float32)
# 
#         # Calculăm trăsăturile geometrice
#         geometric_features = np.array(self.calculate_geometric_features(img_resized), dtype=np.float32)
# 
#         # Combinăm pixelii cu trăsăturile geometrice
#         combined_features = np.hstack((img_flat, geometric_features)).reshape(1, -1)
# 
#         # Predicția modelului
#         prediction = self.model.predict(combined_features)
#         digit = np.argmax(prediction)
#         accuracy = prediction[0][digit] * 100
# 
#         # Afișăm imaginea redimensionată în fereastra Tkinter
#         self.display_image(img_resized)
# 
#         # Actualizăm etichetele cu predicția și acuratețea
#         self.prediction_label.config(text=f'Cifra prezisă: {digit}')
#         self.accuracy_label.config(text=f'Acuratețe: {accuracy:.2f}%')
# 
#         # Afișare în consolă
#         print("Probabilitățile pentru fiecare clasă:", prediction[0])
#         print("Imaginea procesată (28x28):")
#         print(img_resized)
#         print(f'Cifra prezisă: {digit}')
#         print(f'Acuratețe: {accuracy:.2f}%')
# 
#     def display_image(self, img_resized):
#         # Convertim imaginea într-un format compatibil cu Tkinter
#         img_resized_rgb = cv2.cvtColor(img_resized, cv2.COLOR_GRAY2RGB)
#         img_pil = Image.fromarray(img_resized_rgb)
#         img_tk = ImageTk.PhotoImage(img_pil)
# 
#         # Setăm imaginea în Label
#         self.image_label.configure(image=img_tk)
#         self.image_label.image = img_tk  # Păstrăm referința imaginii pentru a preveni eliberarea memoriei
# 
# # Creez fereastra pentru desenare
# root = tk.Tk()
# app = DrawingApp(root)
# root.mainloop()

Nu prea merge, sunt sub 50% predictiile si s si proaste

In [ ]:

# Încarcă setul de date MNIST
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalizează imaginile la intervalul [0, 1]
x_train = tf.keras.utils.normalize(x_train, axis=1)
x_test = tf.keras.utils.normalize(x_test, axis=1)

# Definește funcția pentru calculul trăsăturilor geometrice relevante
def calculate_geometric_features(image):
    moments = cv2.moments(image)
    
    area = moments['m00']
    cx = moments['m10'] / area if area != 0 else 0
    cy = moments['m01'] / area if area != 0 else 0
    Ixx = moments['mu20'] / area if area != 0 else 0
    Iyy = moments['mu02'] / area if area != 0 else 0
    Ixy = moments['mu11'] / area if area != 0 else 0
    angle = 0.5 * np.arctan2(2 * Ixy, Ixx - Iyy)
    
    contours, _ = cv2.findContours(image.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    perimeter = cv2.arcLength(contours[0], True) if contours else 0
    thinness = (4 * np.pi * area) / (perimeter ** 2) if perimeter != 0 else 0
    x, y, w, h = cv2.boundingRect(image.astype(np.uint8))
    elongation = h / w if w != 0 else 0
    
    return [area, cx, cy, angle, perimeter, thinness, elongation]

# Extrage trăsăturile geometrice pentru setul de antrenament și testare
geo_train = np.array([calculate_geometric_features(img) for img in x_train])
geo_test = np.array([calculate_geometric_features(img) for img in x_test])

# Scalează trăsăturile geometrice
scaler = StandardScaler()
geo_train = scaler.fit_transform(geo_train)
geo_test = scaler.transform(geo_test)

# Combină datele de imagine cu trăsăturile geometrice
x_train_combined = np.hstack((x_train.reshape(len(x_train), -1), geo_train))
x_test_combined = np.hstack((x_test.reshape(len(x_test), -1), geo_test))

# Construiește modelul neuronal
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(x_train_combined.shape[1],)),  # Definim inputul explicit
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

# Compilează și antrenează modelul
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train_combined, y_train, epochs=10)

# Evaluăm modelul
test_loss, test_acc = model.evaluate(x_test_combined, y_test)
print(f'Precizia pe setul de testare: {test_acc}')

# Salvăm modelul
model.save('mnist_geometric.keras')


In [ ]:
# # Încarc modelul salvat
# model = tf.keras.models.load_model('mnist_geometric.keras')
# 
# # Funcție pentru calcularea trăsăturilor geometrice
# def calculate_geometric_features(image):
#     moments = cv2.moments(image)
# 
#     # Aria imaginii (numărul de pixeli albi)
#     area = moments['m00']
# 
#     # Calculăm centrul de masă (cx, cy) al imaginii
#     cx = moments['m10'] / area if area != 0 else 0
#     cy = moments['m01'] / area if area != 0 else 0
# 
#     # Calculăm momentul de inerție pentru axa x (Ixx), axa y (Iyy) și momentul mixt (Ixy)
#     Ixx = moments['mu20'] / area if area != 0 else 0
#     Iyy = moments['mu02'] / area if area != 0 else 0
#     Ixy = moments['mu11'] / area if area != 0 else 0
# 
#     # Calculăm unghiul de rotație al cifrei pe baza momentelor
#     angle = 0.5 * np.arctan2(2 * Ixy, Ixx - Iyy)
# 
#     # Calculăm perimetrul conturului cifrei
#     contours, _ = cv2.findContours(image.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#     perimeter = cv2.arcLength(contours[0], True) if contours else 0
# 
#     # Calculăm subțierea (thinness ratio), care este un indicator al formei
#     thinness = (4 * np.pi * area) / (perimeter ** 2) if perimeter != 0 else 0
# 
#     # Calculăm elongarea (aspect ratio) folosind dimensiunile imaginii
#     x, y, w, h = cv2.boundingRect(image.astype(np.uint8))
#     elongation = h / w if w != 0 else 0
# 
#     # Returnăm trăsăturile geometrice relevante
#     return [area, cx, cy, angle, perimeter, thinness, elongation]
# 
# # Preprocesăm trăsăturile geometrice
# scaler = StandardScaler()
# 
# class DrawingApp:
#     def __init__(self, root):
#         self.root = root
#         self.root.title("Desenează o cifră")
# 
#         # Canvas pentru desenat (dimensiune 280x280)
#         self.canvas = tk.Canvas(self.root, width=280, height=280, bg="white")
#         self.canvas.pack()
#         self.canvas.bind("<B1-Motion>", self.paint)
# 
#         # Imaginea stocată de 280x280
#         self.image = np.zeros((280, 280), dtype=np.float32)
# 
#         # Butoane pentru acțiuni
#         self.predict_button = tk.Button(self.root, text="Recunoaște", command=self.recognize_digit)
#         self.predict_button.pack()
#         self.clear_button = tk.Button(self.root, text="Șterge", command=self.clear)
#         self.clear_button.pack()
# 
#         # Etichete pentru a afișa cifra prezisă și acuratețea
#         self.prediction_label = tk.Label(self.root, text="Cifra prezisă: -", font=('Helvetica', 14))
#         self.prediction_label.pack()
#         self.accuracy_label = tk.Label(self.root, text="Acuratețe: -", font=('Helvetica', 14))
#         self.accuracy_label.pack()
# 
#         # Statusul pentru a ști dacă desenăm
#         self.drawing_in_progress = False
# 
#     def paint(self, event):
#         x, y = event.x, event.y
#         # Asigurăm că punctul desenat este în limitele imaginii
#         if 0 <= x < 280 and 0 <= y < 280:
#             self.canvas.create_oval(x-3, y-3, x+3, y+3, fill="black", width=6)  # Pix mai gros
#             self.image[y, x] = 1.0  # Valoare normalizată direct la 1
#             self.drawing_in_progress = True
# 
#     def clear(self):
#         # Resetăm canvas-ul și imaginea pentru a permite un nou desen
#         self.canvas.delete("all")
#         self.image = np.zeros((280, 280), dtype=np.uint8)
#         self.prediction_label.config(text="Cifra prezisă: -")
#         self.accuracy_label.config(text="Acuratețe: -")
#         self.drawing_in_progress = False
# 
#     def recognize_digit(self):
#         if not self.drawing_in_progress:
#             return
# 
#         # Preprocesăm imaginea pentru a fi 28x28
#         img_resized = cv2.resize(self.image, (28, 28))  # Redimensionăm la 28x28
#         img_resized = np.expand_dims(img_resized, axis=-1)  # Adăugăm dimensiunea canalului de culoare
#         # img_resized = img_resized / 255.0  # Normalizăm la [0, 1]
# 
#         # Calculăm trăsăturile geometrice pentru imaginea desenată
#         geo_features = calculate_geometric_features(self.image)
#         geo_features = np.array([geo_features])
#         geo_features_scaled = scaler.fit_transform(geo_features)
# 
#         # Combinăm imaginea cu trăsăturile geometrice
#         img_combined = np.hstack((img_resized.reshape(28 * 28), geo_features_scaled.flatten()))
# 
#         # Predicția modelului
#         prediction = model.predict(np.expand_dims(img_combined, axis=0))  # Adăugăm dimensiunea batch-ului
#         digit = np.argmax(prediction)
# 
#         # Afișăm rezultatul predicției și probabilitățile
#         print(f"Probabilitățile pentru fiecare clasă: {prediction}")
#         self.prediction_label.config(text=f'Cifra prezisă: {digit}')
#         self.accuracy_label.config(text=f'Acuratețe: {prediction[0][digit]*100:.2f}%')
# 
#         # Resetăm statutul de desenare
#         self.drawing_in_progress = False
# 
# # Creez fereastra pentru desenare
# root = tk.Tk()
# app = DrawingApp(root)
# root.mainloop()

Cel mai ok pana acum dar tot praf e si el

In [1]:
model = tf.keras.models.load_model('mnist_geometric2.keras')
class DrawingApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Desenează o cifră")

        # Canvas pentru desenat (dimensiune 280x280 pentru mai mult spațiu)
        self.canvas = tk.Canvas(self.root, width=280, height=280, bg="white")
        self.canvas.pack()
        self.canvas.bind("<B1-Motion>", self.paint)

        # Imaginea stocată (280x280 pentru desen)
        self.image = np.zeros((280, 280), dtype=np.uint8)

        # Butoane pentru acțiuni
        self.predict_button = tk.Button(self.root, text="Recunoaște", command=self.recognize_digit)
        self.predict_button.pack()
        self.clear_button = tk.Button(self.root, text="Șterge", command=self.clear)
        self.clear_button.pack()

        # Etichete pentru a afișa cifra prezisă și acuratețea
        self.prediction_label = tk.Label(self.root, text="Cifra prezisă: -", font=('Helvetica', 14))
        self.prediction_label.pack()
        self.accuracy_label = tk.Label(self.root, text="Acuratețe: -", font=('Helvetica', 14))
        self.accuracy_label.pack()

        # Statusul pentru a ști dacă desenăm
        self.drawing_in_progress = False
        
        # Încarc modelul salvat
        self.model = tf.keras.models.load_model('mnist_geometric.keras')

    def paint(self, event):
        x, y = event.x, event.y
        # Desenăm un cerc la locația curentă
        self.canvas.create_oval(x-5, y-5, x+5, y+5, fill="black", width=5)
        
        # Adăugăm punctul desenat în imagine
        self.image[y, x] = 255  # Setăm culoarea maximă pentru pixel
        self.drawing_in_progress = True

    def clear(self):
        # Resetăm canvas-ul și imaginea pentru a permite un nou desen
        self.canvas.delete("all")
        self.image = np.zeros((280, 280), dtype=np.uint8)
        self.prediction_label.config(text="Cifra prezisă: -")
        self.accuracy_label.config(text="Acuratețe: -")
        self.drawing_in_progress = False

    def recognize_digit(self):
        if not self.drawing_in_progress:
            return

        # Redimensionăm imaginea la 28x28 pentru a se potrivi cu modelul
        img_resized = cv2.resize(self.image, (28, 28))
    
        # Adăugăm zone gri pentru a face imaginea mai detaliată
        img_resized = cv2.GaussianBlur(img_resized, (5, 5), 0)  # Opțional, pentru a face imaginea mai netedă
    
        # Aplatizăm imaginea într-un vector 1D de dimensiune 784
        img_flattened = img_resized.flatten()  # Aplatizează imaginea într-un vector 1D
    
        # Normalizăm valorile pixelilor
        # img_normalized = img_flattened / 255.0  # Normalizarea pentru a fi între 0 și 1
    
        # Adăugăm padding pentru a face dimensiunea 791
        img_padded = np.pad(img_flattened, (0, 791 - 784), mode='constant', constant_values=0)
    
        # Reshape pentru a se potrivi modelului
        img_padded = img_padded.reshape(1, 791)  # Reshape la dimensiunea 791 pentru a fi compatibil cu modelul
    
        # Predicția modelului
        prediction = self.model.predict(img_padded)
        digit = np.argmax(prediction)

        # Afișăm rezultatul predicției și acuratețea
        self.prediction_label.config(text=f'Cifra prezisă: {digit}')
        self.accuracy_label.config(text=f'Acuratețe: {prediction[0][digit]*100:.2f}%')
    
        # Resetăm statutul de desenare
        self.drawing_in_progress = False

# Creez fereastra pentru desenare
root = tk.Tk()
app = DrawingApp(root)
root.mainloop()



KeyboardInterrupt



Incerc cu KNN - prezice doar 1

In [117]:
# Încarc setul de date MNIST
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalizez imaginile la intervalul [0, 1]
x_train = tf.keras.utils.normalize(x_train, axis=1)
x_test = tf.keras.utils.normalize(x_test, axis=1)

# Definirea funcției pentru calculul trăsăturilor geometrice
def calculate_geometric_features(image):
    moments = cv2.moments(image)
    area = moments['m00']
    cx = moments['m10'] / area if area != 0 else 0
    cy = moments['m01'] / area if area != 0 else 0
    Ixx = moments['mu20'] / area if area != 0 else 0
    Iyy = moments['mu02'] / area if area != 0 else 0
    Ixy = moments['mu11'] / area if area != 0 else 0
    angle = 0.5 * np.arctan2(2 * Ixy, Ixx - Iyy)
    
    contours, _ = cv2.findContours(image.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    perimeter = cv2.arcLength(contours[0], True) if contours else 0
    
    thinness = (4 * np.pi * area) / (perimeter ** 2) if perimeter != 0 else 0
    x, y, w, h = cv2.boundingRect(image.astype(np.uint8))
    elongation = h / w if w != 0 else 0
    
    return [area, cx, cy, angle, perimeter, thinness, elongation]

# Extragem trăsăturile geometrice pentru setul de antrenare și testare
geo_train = np.array([calculate_geometric_features(img) for img in x_train])
geo_test = np.array([calculate_geometric_features(img) for img in x_test])

# Scalez trăsăturile geometrice
scaler = StandardScaler()
geo_train = scaler.fit_transform(geo_train)
geo_test = scaler.transform(geo_test)

# Combinăm datele de imagine cu trăsăturile geometrice
x_train_combined = np.hstack((x_train.reshape(len(x_train), -1), geo_train))
x_test_combined = np.hstack((x_test.reshape(len(x_test), -1), geo_test))

# Antrenăm modelul KNN
knn = KNeighborsClassifier(n_neighbors=3)  # Poți ajusta numărul de vecini
knn.fit(x_train_combined, y_train)

# Evaluăm modelul pe setul de testare
test_acc = knn.score(x_test_combined, y_test)
print(f'Precizia pe setul de testare: {test_acc}')

# Salvăm modelul KNN într-un fișier
joblib.dump(knn, 'knn_mnist_model.pkl')

Precizia pe setul de testare: 0.944


['knn_mnist_model.pkl']

In [118]:
# Creez fereastra pentru desenare
class DrawingApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Desenează o cifră")

        # Canvas pentru desenat (dimensiune 280x280 pentru mai mult spațiu)
        self.canvas = tk.Canvas(self.root, width=280, height=280, bg="white")
        self.canvas.pack()
        self.canvas.bind("<B1-Motion>", self.paint)

        # Imaginea stocată (280x280 pentru desen)
        self.image = np.zeros((280, 280), dtype=np.uint8)

        # Butoane pentru acțiuni
        self.predict_button = tk.Button(self.root, text="Recunoaște", command=self.recognize_digit)
        self.predict_button.pack()
        self.clear_button = tk.Button(self.root, text="Șterge", command=self.clear)
        self.clear_button.pack()

        # Etichete pentru a afișa cifra prezisă și acuratețea
        self.prediction_label = tk.Label(self.root, text="Cifra prezisă: -", font=('Helvetica', 14))
        self.prediction_label.pack()
        self.accuracy_label = tk.Label(self.root, text="Acuratețe: -", font=('Helvetica', 14))
        self.accuracy_label.pack()

        # Statusul pentru a ști dacă desenăm
        self.drawing_in_progress = False

        # Încarc modelul KNN salvat
        self.model = joblib.load('knn_mnist_model.pkl')

    def paint(self, event):
        x, y = event.x, event.y
        # Desenăm un cerc la locația curentă
        self.canvas.create_oval(x-5, y-5, x+5, y+5, fill="black", width=5)
        
        # Adăugăm punctul desenat în imagine
        self.image[y, x] = 255  # Setăm culoarea maximă pentru pixel
        self.drawing_in_progress = True

    def clear(self):
        # Resetăm canvas-ul și imaginea pentru a permite un nou desen
        self.canvas.delete("all")
        self.image = np.zeros((280, 280), dtype=np.uint8)
        self.prediction_label.config(text="Cifra prezisă: -")
        self.accuracy_label.config(text="Acuratețe: -")
        self.drawing_in_progress = False

    def recognize_digit(self):
        if not self.drawing_in_progress:
            return

        # Redimensionăm imaginea la 28x28 pentru a se potrivi cu modelul
        img_resized = cv2.resize(self.image, (28, 28))
        img_resized = cv2.GaussianBlur(img_resized, (5, 5), 0)  # Opțional, pentru a face imaginea mai netedă

        # Binarizare imagine
        _, img_binarized = cv2.threshold(img_resized, 127, 255, cv2.THRESH_BINARY)
        img_normalized = img_binarized / 255.0

        # Calculăm trăsăturile geometrice
        geo_features = calculate_geometric_features(img_normalized)

        # Redimensionăm și combinăm cu trăsăturile geometrice
        img_flat = img_normalized.reshape(1, -1)
        img_combined = np.hstack((img_flat, np.array(geo_features).reshape(1, -1)))

        # Predicția cu KNN
        prediction = self.model.predict(img_combined)

        # Afișăm rezultatul predicției
        self.prediction_label.config(text=f'Cifra prezisă: {prediction[0]}')

        # Calculăm acuratețea (deocamdată nu avem eticheta reală, dar putem folosi o valoare simulată)
        # Dacă ai un set de etichete reale, le poți compara cu predicțiile pentru a obține acuratețea reală
        self.accuracy_label.config(text=f'Acuratețe: -')

        self.drawing_in_progress = False

# Creez fereastra pentru desenare
root = tk.Tk()
app = DrawingApp(root)
root.mainloop()